# Model Evaluation & Comparison

Reads pre-computed metrics from every model notebook and produces a single cross-model,
cross-frequency leaderboard. Run after all model notebooks have been executed.

**Models covered**

| Model | Weekly | Daily |
|---|:---:|:---:|
| ARIMA / ARIMAX | ✓ | ✓ |
| VAR | ✓ | ✓ |
| Random Forest | ✓ | ✓ |
| XGBoost | ✓ | ✓ |
| LSTM | ✓ | — |
| MIDAS | ✓ | — |

**Primary metric**: WDA (Weighted Directional Accuracy) — weights correct direction calls
by the magnitude of the actual return. Secondary: DA, RMSE, MAE.

In [ ]:
import sys, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

DATA = '../data/processed/'

## 1. Load & harmonise metrics

In [ ]:
# file → (frequency, model family)
metrics_map = {
    'metrics_arima.csv':         ('Weekly', 'ARIMA'),
    'metrics_var.csv':           ('Weekly', 'VAR'),
    'metrics_rf.csv':            ('Weekly', 'Random Forest'),
    'metrics_xgboost.csv':       ('Weekly', 'XGBoost'),
    'metrics_lstm.csv':          ('Weekly', 'LSTM'),
    'metrics_midas.csv':         ('Weekly', 'MIDAS'),
    'metrics_arima_daily.csv':   ('Daily',  'ARIMA'),
    'metrics_var_daily.csv':     ('Daily',  'VAR'),
    'metrics_rf_daily.csv':      ('Daily',  'Random Forest'),
    'metrics_xgboost_daily.csv': ('Daily',  'XGBoost'),
}

frames = []
for fname, (freq, family) in metrics_map.items():
    path = DATA + fname
    if not os.path.exists(path):
        print(f'Missing: {fname}')
        continue
    df = pd.read_csv(path)
    # normalise column names
    df = df.rename(columns={'dir_acc': 'da'})
    df['frequency'] = freq
    df['family']    = family
    frames.append(df)

all_metrics = pd.concat(frames, ignore_index=True)

# Flag naive / baseline rows
all_metrics['is_naive'] = all_metrics['model'].str.lower().str.startswith('naive')

print(f'Loaded {len(all_metrics)} rows across {all_metrics["family"].nunique()} model families')
all_metrics.head(10)

## 2. Full leaderboard

Best variant per model family (highest WDA), ranked within each frequency.

In [ ]:
models_only = all_metrics[~all_metrics['is_naive']].copy()

# Best variant per family × frequency
best = (
    models_only
    .sort_values('wda', ascending=False)
    .groupby(['frequency', 'family'], sort=False)
    .first()
    .reset_index()
    .sort_values(['frequency', 'wda'], ascending=[True, False])
)

for freq in ['Weekly', 'Daily']:
    sub = best[best['frequency'] == freq][['family', 'model', 'rmse', 'mae', 'da', 'wda']].copy()
    sub = sub.rename(columns={'family': 'Model', 'model': 'Best variant'})
    sub[['rmse', 'mae']] = sub[['rmse', 'mae']].round(4)
    sub[['da', 'wda']]   = sub[['da', 'wda']].round(3)
    print(f'\n── {freq} ──')
    print(sub.to_string(index=False))

## 3. Visual comparison — WDA & DA by model

In [ ]:
def plot_metric_bars(ax, df_freq, metric, title, naive_val=None):
    df = df_freq.sort_values(metric, ascending=True)
    colors = ['#2ca02c' if v > (naive_val or 0.5) else '#d62728'
              for v in df[metric]]
    ax.barh(df['model'], df[metric], color=colors, alpha=0.85)
    if naive_val:
        ax.axvline(naive_val, color='black', lw=1.2, ls='--', label=f'Naive ({naive_val:.3f})')
    ax.axvline(0.5, color='grey', lw=0.8, ls=':', label='50% (coin flip)')
    ax.set_xlabel(metric.upper())
    ax.set_title(title)
    ax.legend(fontsize=8)
    for v, name in zip(df[metric], df['model']):
        ax.text(v + 0.002, name, f'{v:.3f}', va='center', fontsize=8)

for freq in ['Weekly', 'Daily']:
    sub     = models_only[models_only['frequency'] == freq]
    naive   = all_metrics[
        (all_metrics['frequency'] == freq) & all_metrics['is_naive']
    ]['wda'].mean()

    fig, axes = plt.subplots(1, 2, figsize=(14, max(4, len(sub) * 0.35 + 2)))
    fig.suptitle(f'{freq} models — directional accuracy', fontweight='bold')

    plot_metric_bars(axes[0], sub, 'wda', 'WDA (primary)', naive_val=naive)
    plot_metric_bars(axes[1], sub, 'da',  'DA  (secondary)', naive_val=naive)

    plt.tight_layout()
    plt.show()

## 4. RMSE / MAE comparison

In [ ]:
for freq in ['Weekly', 'Daily']:
    sub = models_only[models_only['frequency'] == freq].sort_values('rmse')

    fig, axes = plt.subplots(1, 2, figsize=(14, max(4, len(sub) * 0.35 + 2)))
    fig.suptitle(f'{freq} models — point-forecast error', fontweight='bold')

    for ax, metric in zip(axes, ['rmse', 'mae']):
        df = sub.sort_values(metric, ascending=True)
        ax.barh(df['model'], df[metric], color='steelblue', alpha=0.85)
        ax.set_xlabel(metric.upper())
        ax.set_title(metric.upper())
        for v, name in zip(df[metric], df['model']):
            ax.text(v * 1.005, name, f'{v:.4f}', va='center', fontsize=8)

    plt.tight_layout()
    plt.show()

## 5. Cross-frequency comparison

For models with both daily and weekly variants: does aggregating to weekly
lose or gain predictive signal?

In [ ]:
both_freq = set(models_only[models_only['frequency'] == 'Weekly']['family']) & \
            set(models_only[models_only['frequency'] == 'Daily']['family'])

cross = (
    models_only[models_only['family'].isin(both_freq)]
    .sort_values('wda', ascending=False)
    .groupby(['family', 'frequency'])
    .first()
    .reset_index()[['family', 'frequency', 'model', 'wda', 'da', 'rmse']]
)

# Pivot for easy comparison
pivot = cross.pivot(index='family', columns='frequency', values=['wda', 'da', 'rmse'])
pivot.columns = [f'{m}_{f.lower()}' for m, f in pivot.columns]
pivot['wda_diff'] = pivot['wda_weekly'] - pivot['wda_daily']
pivot = pivot.sort_values('wda_diff', ascending=False)
print('WDA: weekly vs daily (positive = weekly wins)')
print(pivot[['wda_weekly', 'wda_daily', 'wda_diff']].round(3).to_string())

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(pivot))
w = 0.35
ax.bar(x - w/2, pivot['wda_weekly'], w, label='Weekly', color='steelblue', alpha=0.85)
ax.bar(x + w/2, pivot['wda_daily'],  w, label='Daily',  color='darkorange', alpha=0.85)
ax.axhline(0.5, color='grey', lw=0.8, ls=':', label='50%')
ax.set_xticks(x)
ax.set_xticklabels(pivot.index)
ax.set_ylabel('WDA')
ax.set_title('Best-variant WDA: Weekly vs Daily')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Sentiment ablation

Does adding retail sentiment improve forecasts?

**ARIMA**: baseline (no exogenous) vs ARIMAX variants (market covariates, Reddit, News, Trends).  
**LSTM**: Y-only vs EXOG (market) vs REDDIT vs NEWS.

In [ ]:
arima_w = all_metrics[
    (all_metrics['family'] == 'ARIMA') &
    (all_metrics['frequency'] == 'Weekly') &
    ~all_metrics['is_naive']
].copy()

lstm_w = all_metrics[
    (all_metrics['family'] == 'LSTM') &
    ~all_metrics['is_naive']
].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Sentiment ablation — WDA', fontweight='bold')

for ax, df, title in [
    (axes[0], arima_w, 'ARIMA / ARIMAX (weekly)'),
    (axes[1], lstm_w,  'LSTM variants (weekly)'),
]:
    df_s = df.sort_values('wda', ascending=True)
    baseline_wda = df_s.iloc[0]['wda']
    colors = ['#2ca02c' if v > baseline_wda else '#d62728'
              for v in df_s['wda']]
    ax.barh(df_s['model'], df_s['wda'], color=colors, alpha=0.85)
    ax.axvline(0.5, color='grey', lw=0.8, ls=':', label='50%')
    ax.set_xlabel('WDA')
    ax.set_title(title)
    ax.legend(fontsize=8)
    for v, name in zip(df_s['wda'], df_s['model']):
        ax.text(v + 0.002, name, f'{v:.3f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

## 7. MIDAS overview

MIDAS uses monthly macro data (CPI, industrial production, fed funds, real rates)
mixed with the weekly silver return series. Shown separately because it operates
on a shorter test window than the other weekly models.

In [ ]:
midas = all_metrics[all_metrics['family'] == 'MIDAS'].copy()
midas_s = midas.sort_values('wda', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, max(4, len(midas) * 0.4 + 2)))
fig.suptitle('MIDAS variants', fontweight='bold')

for ax, metric in zip(axes, ['wda', 'rmse']):
    df = midas_s.sort_values(metric, ascending=(metric == 'rmse'))
    ax.barh(df['model'], df[metric], color='steelblue', alpha=0.85)
    if metric == 'wda':
        ax.axvline(0.5, color='grey', lw=0.8, ls=':', label='50%')
        ax.legend(fontsize=8)
    ax.set_xlabel(metric.upper())
    ax.set_title(metric.upper())
    for v, name in zip(df[metric], df['model']):
        offset = 0.002 if metric == 'wda' else df[metric].max() * 0.005
        ax.text(v + offset, name, f'{v:.4f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

## 8. Silver Squeeze — LSTM episode analysis

LSTM variants tested specifically on the Jan–Jun 2021 squeeze episode.
Does the model that saw Reddit/news sentiment do better during the peak retail event?

In [ ]:
squeeze_path = DATA + 'metrics_lstm_squeeze.csv'
if not os.path.exists(squeeze_path):
    print('metrics_lstm_squeeze.csv not found — run weekly/07_lstm_squeeze.ipynb first.')
else:
    squeeze = pd.read_csv(squeeze_path)
    print(squeeze.columns.tolist())

    periods = squeeze['period'].unique()
    n_periods = len(periods)

    fig, axes = plt.subplots(1, n_periods, figsize=(7 * n_periods, 5), sharey=False)
    if n_periods == 1:
        axes = [axes]
    fig.suptitle('LSTM — Silver Squeeze episode WDA by sub-period', fontweight='bold')

    for ax, period in zip(axes, periods):
        sub = squeeze[squeeze['period'] == period].sort_values('wda', ascending=True)
        colors = ['#2ca02c' if v > 0.5 else '#d62728' for v in sub['wda']]
        ax.barh(sub['model'], sub['wda'], color=colors, alpha=0.85)
        ax.axvline(0.5, color='grey', lw=0.8, ls=':', label='50%')
        ax.set_xlabel('WDA')
        ax.set_title(period, fontsize=9)
        ax.legend(fontsize=8)
        for v, name in zip(sub['wda'], sub['model']):
            ax.text(v + 0.003, name, f'{v:.3f}', va='center', fontsize=8)

    plt.tight_layout()
    plt.show()
    print(squeeze.to_string(index=False))

## 9. Diebold-Mariano tests — ARIMA variants

Tests whether ARIMAX variants are significantly better or worse than the ARIMA baseline.
Uses saved prediction files (`preds_*.csv`); squared-error loss, Newey-West variance (lag=1).

In [ ]:
def diebold_mariano(actual, pred1, pred2, name1='Model 1', name2='Model 2'):
    e1 = actual - pred1
    e2 = actual - pred2
    d  = e1**2 - e2**2
    T  = len(d)
    d_bar = d.mean()
    # Newey-West variance with lag 1
    gamma0 = np.mean((d - d_bar)**2)
    gamma1 = np.mean((d[1:] - d_bar) * (d[:-1] - d_bar))
    nw_var = (gamma0 + 2 * gamma1) / T
    dm_stat = d_bar / np.sqrt(max(nw_var, 1e-12))
    p_val   = 2 * (1 - stats.norm.cdf(abs(dm_stat)))
    winner  = name1 if dm_stat < 0 else name2
    sig     = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else ''
    print(f'  {name1:<30} vs {name2:<30}  DM={dm_stat:+.3f}  p={p_val:.3f}{sig}  better={winner}')
    return {'name1': name1, 'name2': name2, 'dm': dm_stat, 'p': p_val, 'winner': winner}

# Load ARIMA baseline predictions
preds_files = {
    'ARIMA rolling':        'preds_arima_rol.csv',
    'ARIMAX rolling':       'preds_arimax_rol.csv',
    'ARIMAX + Reddit':      'preds_arimax_reddit_rol.csv',
    'ARIMAX + News':        'preds_arimax_news_rol.csv',
    'ARIMAX + Trends':      'preds_arimax_trends_rol.csv',
    'ARIMAX + Reddit+News': 'preds_arimax_both_rol.csv',
    'ARIMAX + R+N+Trends':  'preds_arimax_reddit_trends_rol.csv',
}

preds = {}
for name, fname in preds_files.items():
    path = DATA + fname
    if os.path.exists(path):
        df = pd.read_csv(path, index_col=0, parse_dates=True)
        preds[name] = df
    else:
        print(f'Missing: {fname}')

print(f'Loaded {len(preds)} prediction series\n')

if 'ARIMA rolling' in preds:
    baseline = preds['ARIMA rolling']
    actual   = baseline['actual'].values
    pred_b   = baseline['predicted'].values

    print('H₀: equal predictive accuracy vs ARIMA rolling baseline')
    print('  Negative DM → baseline is better; Positive DM → challenger is better\n')
    results = []
    for name, df in preds.items():
        if name == 'ARIMA rolling':
            continue
        common = baseline.index.intersection(df.index)
        r = diebold_mariano(
            baseline.loc[common, 'actual'].values,
            pred_b[:len(common)],
            df.loc[common, 'predicted'].values,
            name1='ARIMA rolling',
            name2=name,
        )
        results.append(r)
    print('\n* p<0.05  ** p<0.01  *** p<0.001')

## 10. Summary table — thesis-ready

One row per model family × frequency showing the best-variant metrics.

In [ ]:
summary = (
    best[['frequency', 'family', 'model', 'rmse', 'mae', 'da', 'wda']]
    .rename(columns={
        'frequency': 'Frequency',
        'family':    'Model',
        'model':     'Best variant',
        'rmse':      'RMSE',
        'mae':       'MAE',
        'da':        'DA',
        'wda':       'WDA',
    })
    .sort_values(['Frequency', 'WDA'], ascending=[True, False])
)

summary[['RMSE', 'MAE']] = summary[['RMSE', 'MAE']].round(4)
summary[['DA', 'WDA']]   = summary[['DA', 'WDA']].round(3)

print(summary.to_string(index=False))

# LaTeX
latex = summary.to_latex(index=False, float_format='%.3f',
                          caption='Model comparison — best variant per family',
                          label='tab:model_comparison')
print('\n── LaTeX ──')
print(latex)

## 11. Period breakdown — WDA by calendar year

Splits each model's best-variant predictions into sub-periods to check whether
good overall WDA is consistent across market regimes or driven by one lucky stretch.

Periods: 2023 (choppy), 2024 (bull start), 2025 (bull run), 2026 YTD.

In [ ]:
period_map = {
    'ARIMA Weekly':  'period_arima_weekly.csv',
    'VAR Weekly':    'period_var_weekly.csv',
    'RF Weekly':     'period_rf_weekly.csv',
    'XGB Weekly':    'period_xgboost_weekly.csv',
    'ARIMA Daily':   'period_arima_daily.csv',
    'VAR Daily':     'period_var_daily.csv',
    'RF Daily':      'period_rf_daily.csv',
    'XGB Daily':     'period_xgboost_daily.csv',
}

period_frames = {}
for label, fname in period_map.items():
    path = DATA + fname
    if os.path.exists(path):
        period_frames[label] = pd.read_csv(path, index_col=0)
    else:
        print(f'Missing: {fname} — run the model notebook first')

if not period_frames:
    print('No period files found. Re-run model notebooks to generate them.')
else:
    # Build WDA matrix: rows=model, columns=period
    wda_matrix = pd.DataFrame({lbl: df['WDA'] for lbl, df in period_frames.items()}).T
    wda_matrix.index.name = 'Model'

    full_col  = '── Full test ──'
    year_cols = [c for c in wda_matrix.columns if c != full_col]

    # ── heatmap (year sub-periods only) ───────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.heatmap(
        wda_matrix[year_cols],
        annot=True, fmt='.3f', cmap='RdYlGn',
        vmin=0.35, vmax=0.70, center=0.50,
        linewidths=0.5, ax=ax,
    )
    ax.set_title('WDA by model and sub-period', fontweight='bold')
    ax.set_ylabel('')
    plt.tight_layout()
    plt.show()

    # ── consistency table ─────────────────────────────────────────────────────
    consistency = pd.DataFrame({
        'Full WDA':  wda_matrix[full_col] if full_col in wda_matrix.columns else np.nan,
        'Mean (years)': wda_matrix[year_cols].mean(axis=1),
        'Std (years)':  wda_matrix[year_cols].std(axis=1),
        'Min (year)':   wda_matrix[year_cols].min(axis=1),
        'Max (year)':   wda_matrix[year_cols].max(axis=1),
    }).sort_values('Full WDA', ascending=False)

    print('Consistency across sub-periods (lower Std = more stable):')
    display(consistency.style
            .format('{:.3f}')
            .background_gradient(cmap='RdYlGn', subset=['Full WDA', 'Mean (years)'], vmin=0.38, vmax=0.68)
            .background_gradient(cmap='RdYlGn_r', subset=['Std (years)'], vmin=0.0, vmax=0.15))
